In [ ]:
import numpy as np
from astropy.io import fits
import pandas as pd
from astropy.table import Table
import matplotlib.pyplot as plt
import os
from metrics_newlcs_oct26 import info

ModuleNotFoundError: No module named 'metrics_newlcs_oct26'

In [2]:
plt.rcParams['font.family'] = 'serif'

In [12]:
path = "./topcat_contamination_may10"
path = "./topcat_contamination_oct25"
# table_names = ['deltascut_lmc', 'doublepvar_lmc', 'typeIIcepheids_lmc', 'classicalcepheids_lmc', 
#                'RRlyrae_lmc', 'lpv_lmc', 'classicalcepheids_smc',
#                'RRlyrae_smc', 'eb_lmc', 'lpv_smc', 'eb_smc'] #'lbv_smc', 

table_names = ['typeIIcepheids_lmc', 'deltascut_lmc', 'classicalcepheids_lmc', 
               'RRlyrae_lmc', 'lpv_lmc', 'lpv_smc', 'eb_smc', 'classicalcepheids_smc',
               'RRlyrae_smc', 'eb_lmc']

with fits.open(path) as hdul:
    for name, hdu in zip(table_names, hdul[2:]):
        print(
            hdu.index if hasattr(hdu, "index") else "",
            hdu.header.get("EXTNAME"),
            hdu.header.get("NAXIS2"),
            name,
        )

 1xJ/AcA/68/89/lmc_t2cep 2 typeIIcepheids_lmc
 1xJ/AcA/60/1/catalog 10 deltascut_lmc
 1xJ/AcA/58/163/catalog 28 classicalcepheids_lmc
 1xJ/AcA/59/1/catalog 36 RRlyrae_lmc
 1xJ/AcA/59/239/catalog 145 lpv_lmc
 1xJ/AcA/61/217/lpv 112 lpv_smc
 1xJ/AcA/63/323/ecl 74 eb_smc
 1xJ/AcA/60/17/catalog 52 classicalcepheids_smc
 1xJ/AcA/60/165/ident 16 RRlyrae_smc
 1xJ/AcA/61/103/ecl 95 eb_lmc


In [23]:
tolerance = 0.05
min_angdist = 2.0
flagged_overall = 0

table_labels = [
    "typeIIcepheids_lmc",
    "deltascut_lmc",
    "classicalcepheids_lmc",
    "RRlyrae_lmc",
    "lpv_lmc",
    "lpv_smc",
    "eb_smc",
    "classicalcepheids_smc",
    "RRlyrae_smc",
    "eb_lmc",
]

flagged_entries = []

with fits.open(path) as hdul:
    crossmatch_hdus = hdul[2:]

    if len(crossmatch_hdus) != len(table_labels):
        raise ValueError(
            f"Found {len(crossmatch_hdus)} crossmatch HDUs, "
            f"but {len(table_labels)} table labels were provided."
        )

    for table_label, hdu in zip(table_labels, crossmatch_hdus):
        hdu_index = hdul.index(hdu)
        table_name = hdu.header.get("EXTNAME", f"HDU_{hdu_index}")
        table = Table.read(path, hdu=hdu_index)

        required_cols = {"best_period", "Per", "angDist", "star_idx"}
        missing = required_cols - set(table.colnames)

        if missing:
            required_cols = {"best_period", "Per1", "angDist", "star_idx"}
            missing = required_cols - set(table.colnames)
            if missing:
                print(f"Skipping {table_name} ({table_label}): "
                      f"missing columns {sorted(missing)}")
                continue
            #rename per1 to per
            table.rename_column("Per1", "Per")
            # print(f"Skipping {table_name} ({table_label}): "
            #       f"missing columns {sorted(missing)}")
            # continue

        flagged_rows = 0

        print(f"-------Processing {table_name} ({table_label})-----")

        for row in table:
            best_period = row["best_period"]
            per = row["Per"]
            ang_dist = row["angDist"]

            if ang_dist < min_angdist:
                continue

            period_difference = abs(per - best_period)
            allowed_difference = tolerance * abs(best_period)

            if period_difference <= allowed_difference:
                print(
                    f"FLAG | star_idx={row['star_idx']} "
                    f"({row.get('RA', 'NA')},{row.get('DEC', 'NA')}) | "
                    f"angDist={ang_dist:.6g} | "
                    f"best_period={best_period:.6g} | "
                    f"Per={per:.6g}"
                )

                flagged_rows += 1
                flagged_overall += 1

                flagged_entries.append((
                    table_name,
                    table_label,
                    int(row["star_idx"]),
                    float(row.get("RA", np.nan)),
                    float(row.get("DEC", np.nan)),
                    float(ang_dist),
                    float(best_period),
                    float(per),
                    int(hdu_index),
                ))

        print(
            f"Done {table_name} {table_label}: "
            f"{flagged_rows} flagged row(s) "
            f"with angDist >= {min_angdist}"
        )

print(f"Total flagged rows: {flagged_overall}")

if flagged_entries:
    from astropy.table import Table as AstTable

    col_names = [
        "table_name",
        "table_label",
        "star_idx",
        "ra",
        "dec",
        "angDist",
        "best_period",
        "Per",
        "hdu_index",
    ]

    flagged_table = AstTable(rows=flagged_entries, names=col_names)
    out_path = "./contaminated_flagged_stars_oct26.csv"
    flagged_table.write(out_path, format="csv", overwrite=True)
    print(f"Wrote {len(flagged_entries)} flagged entries to {out_path}")
else:
    print("No flagged entries found; no CSV written.")

-------Processing 1xJ/AcA/68/89/lmc_t2cep (typeIIcepheids_lmc)-----
Done 1xJ/AcA/68/89/lmc_t2cep typeIIcepheids_lmc: 0 flagged row(s) with angDist >= 2.0
-------Processing 1xJ/AcA/60/1/catalog (deltascut_lmc)-----
Done 1xJ/AcA/60/1/catalog deltascut_lmc: 0 flagged row(s) with angDist >= 2.0
-------Processing 1xJ/AcA/58/163/catalog (classicalcepheids_lmc)-----
FLAG | star_idx=395 (72.485769,-69.755859) | angDist=15.3439 | best_period=27.1165 | Per=27.1006
FLAG | star_idx=399 (72.593866,-69.332047) | angDist=13.8071 | best_period=2.23887 | Per=2.23906
FLAG | star_idx=444 (74.058839,-67.413483) | angDist=8.88647 | best_period=3.60558 | Per=3.60559
FLAG | star_idx=495 (75.158007,-67.764633) | angDist=18.506 | best_period=3.52438 | Per=3.52434
FLAG | star_idx=829 (86.675929,-68.008675) | angDist=7.49653 | best_period=2.56664 | Per=2.56587
Done 1xJ/AcA/58/163/catalog classicalcepheids_lmc: 5 flagged row(s) with angDist >= 2.0
-------Processing 1xJ/AcA/59/1/catalog (RRlyrae_lmc)-----
Done 1xJ

In [5]:
unflagged = pd.read_csv("./ysgs_cut_without_contaminants.csv")
mike_spectra = unflagged[unflagged["scope"] == "magellan_drout"]
gemini_candidates = unflagged[unflagged["gemini_candidates"] == 1]


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# ax.scatter(unflagged["overall_median_mag"], unflagged["mag_amplitude_1"], s=10, color="blue", alpha=0.5, label="Unflagged Stars")
ax.scatter(mike_spectra["overall_median_mag"], mike_spectra["mag_amplitude_1"], s=20, color="orange", alpha=0.7, label="Mike Spectra")
ax.scatter(gemini_candidates["overall_median_mag"], gemini_candidates["mag_amplitude_1"], s=20, color="green", alpha=0.7, label="Gemini Candidates")
ax.scatter(unflagged["overall_median_mag"], unflagged["mag_amplitude_1"], s=20, color="blue", alpha=0.5, label="YSGs", zorder=0)
ax.set_xlabel("Median Magnitude", fontsize=16)
ax.set_ylabel("Amplitude", fontsize=16)
ax.legend(fontsize=15)
plt.savefig("./contamination_removed_plot.png", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
# ax.scatter(unflagged["overall_median_mag"], unflagged["mag_amplitude_1"], s=10, color="blue", alpha=0.5, label="Unflagged Stars")
priority_labels = unflagged["priority"].fillna("missing").astype(str)
priority_order = list(dict.fromkeys(priority_labels))
palette = plt.cm.tab10(np.linspace(0, 1, max(len(priority_order), 1)))
priority_color_map = dict(zip(priority_order, palette))
priority_colors = priority_labels.map(priority_color_map)

ax.scatter(
    unflagged["overall_median_mag"],
    unflagged["mag_amplitude_1"],
    s=20,
    c=priority_colors,
    alpha=0.5,
    label="YSGs",
    zorder=0,
)
ax.set_xlabel("Median Magnitude", fontsize=16)
ax.set_ylabel("Amplitude", fontsize=16)
legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=priority_color_map[label], markersize=8, label=label)
    for label in priority_order
]
ax.legend(handles=legend_handles, title="priority", fontsize=12, title_fontsize=13)
plt.savefig("./contamination_removed_priority_plot.png", dpi=300)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

flag_columns = [
    "unknown",
    "lc_issues",
    "two_dips",
    "big_spike",
    "eclipsing_binary_cands",
    "large_spikes",
    "cepheidlike_regular",
    "spikes_irregular",
    "long_trend"
]

colors = plt.cm.tab10(np.linspace(0, 1, len(flag_columns)))

for zorder, (column, color) in enumerate(zip(flag_columns, colors), start=1):
    if column not in unflagged.columns:
        continue

    mask = unflagged[column].fillna(0).eq(1)
    if not mask.any():
        continue

    ax.scatter(
        unflagged.loc[mask, "overall_median_mag"],
        unflagged.loc[mask, "mag_amplitude_1"],
        s=20,
        color=color,
        alpha=0.7,
        label=column,
        zorder=zorder,
    )

ax.set_xlabel("Median Magnitude", fontsize=16)
ax.set_ylabel("Amplitude", fontsize=16)
ax.legend(fontsize=12, title="flags", loc="center left", bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.savefig("./contamination_removed_plot_classified.png", dpi=300)

In [ ]:
info(965)

In [ ]:
print(len(unflagged[unflagged['cepheidlike_regular']==1]))
for star in unflagged[unflagged['cepheidlike_regular']==1]['star_idx']:
    info(star)


In [ ]:
# info(1103) # some telescopes are weird on the lc for all these. clearly super bright
# info(1261)
# info(965)
# info(1115)
# info(1253) #except this one, more minor issues
# info(1174)
# info(864) # no issues with this one
info(1244)


In [17]:
# # from flagged_stars_and_cephs_removed.csv, remove any stars for which 'overall_median_mag' is nan and re-save the file

# df = pd.read_csv('flagged_stars_removed.csv')
# df = df.dropna(subset=['overall_median_mag'])
# df.to_csv('flagged_stars_removed.csv', index=False)

# df = pd.read_csv('flagged_stars_and_cephs_removed.csv')
# df = df.dropna(subset=['overall_median_mag'])
# df.to_csv('flagged_stars_and_cephs_removed.csv', index=False)


In [18]:
# Add gemini_candidates column to flagged_stars_and_cephs_removed.csv
chosen = [40, 156, 250, 307, 522, 613, 716, 780, 890, 953, 1085, 1130]

df = pd.read_csv('./ysgs_cut_without_contaminants.csv')

# 0 by default
df['gemini_candidates'] = 0

# Mark rows where star_idx is in the chosen list
df.loc[df['star_idx'].isin(chosen), 'gemini_candidates'] = 1

df.to_csv('./ysgs_cut_without_contaminants.csv', index=False)

In [56]:
# Crossmatch ysgs_cut_without_contaminants.csv against ../offsets.csv for discarded light curves only
ysgs = pd.read_csv('./ysgs_cut_without_contaminants.csv')
offsets = pd.read_csv('../offsets.csv')

ysgs['star_idx'] = pd.to_numeric(ysgs['star_idx'], errors='coerce').astype('Int64')
offsets['target'] = pd.to_numeric(offsets['target'], errors='coerce').astype('Int64')
offsets['discard_lc'] = pd.to_numeric(offsets['discard_lc'], errors='coerce')

offsets_discarded = offsets[offsets['discard_lc'] == 1]
matched = ysgs.merge(
    offsets_discarded[['target', 'ra', 'dec', 'total_offsets', 'offset_telescopes', 'discard_telescopes', 'visual_check', 'discard_lc']],
    left_on='star_idx',
    right_on='target',
    how='inner',
    suffixes=('_ysgs', '_offsets'),
)

matched_star_idx = sorted(matched['star_idx'].dropna().astype(int).unique().tolist())
print('Matched star_idx values with discard_lc == 1:')
print(matched_star_idx)
print(f'Total matches: {len(matched_star_idx)}')

matched[['star_idx', 'ra_ysgs', 'dec_ysgs', 'ra_offsets', 'dec_offsets', 'total_offsets', 'offset_telescopes', 'discard_telescopes', 'visual_check', 'discard_lc']].head(20)

Matched star_idx values with discard_lc == 1:
[1244]
Total matches: 1


,star_idx,ra_ysgs,dec_ysgs,ra_offsets,dec_offsets,total_offsets,offset_telescopes,discard_telescopes,visual_check,discard_lc
0,1244,85.240855,-69.258659,85.240855,-69.258659,NaN,NaN,NaN,NaN,1.0


In [ ]:
ysgs = pd.read_csv('ysgs_cut_without_contaminants.csv')

#median mag vs amplitude plot

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(ysgs["overall_median_mag"], ysgs["mag_amplitude_1"], s=20, color="blue", alpha=0.5, label="YSGs")
ax.set_xlabel("Median Magnitude", fontsize=16)
ax.set_ylabel("Amplitude", fontsize=16)
ax.legend(fontsize=15)
plt.savefig("../figs/misc/ysgs_median_mag_vs_amplitude.png", dpi=300)

In [ ]:
cephs = pd.read_csv('flagged_stars_removed.csv')
cephs = cephs[cephs['SIMBAD_maintype'] == 'ClassicalCep']
fig, ax = plt.subplots(figsize=(8, 6))
#period luminosity plot for cephs
ax.scatter(cephs["best_period"], cephs["logL"], s=20, color="red", alpha=0.5, label="Classical Cepheids")
ax.set_xlabel("Best Period", fontsize=16)
ax.set_ylabel("Log Luminosity", fontsize=16)
ax.legend(fontsize=15)
ax.set_xscale('log')
plt.savefig("../figs/misc/cephs_period_luminosity.png", dpi=300)

# may 2026

In [ ]:
# -------Processing 1xJ/AcA/58/163/catalog (classicalcepheids_lmc)-----
# FLAG | star_idx=395 (72.485769,-69.755859) | angDist=15.3439 | best_period=27.106 | Per=27.1006
# FLAG | star_idx=444 (74.058839,-67.413483) | angDist=8.88647 | best_period=3.60578 | Per=3.60559
# FLAG | star_idx=495 (75.158007,-67.764633) | angDist=18.506 | best_period=3.52415 | Per=3.52434
# FLAG | star_idx=790 (85.465791,-70.423386) | angDist=19.4022 | best_period=4.55035 | Per=4.55057
# FLAG | star_idx=829 (86.675929,-68.008675) | angDist=7.49653 | best_period=2.56647 | Per=2.56587
# FLAG | star_idx=839 (88.22397,-69.48793) | angDist=18.6231 | best_period=9.1275 | Per=9.12537
# FLAG | star_idx=843 (89.45843,-68.462112) | angDist=4.08578 | best_period=4.45124 | Per=4.44968
# FLAG | star_idx=1068 (76.077504,-68.926216) | angDist=15.7529 | best_period=20.6395 | Per=20.6394
# Done 1xJ/AcA/58/163/catalog classicalcepheids_lmc: 8 flagged row(s) with angDist >= 2.0

# -------Processing 1xJ/AcA/60/17/catalog (classicalcepheids_smc)-----
# FLAG | star_idx=44 (11.955413,-73.095612) | angDist=15.1352 | best_period=2.58519 | Per=2.58496
# FLAG | star_idx=54 (12.144304,-73.154884) | angDist=2.41889 | best_period=1.45451 | Per=1.45443
# FLAG | star_idx=65 (12.419711,-73.236916) | angDist=5.97339 | best_period=15.6625 | Per=15.6574
# FLAG | star_idx=66 (12.42137,-72.301399) | angDist=10.1146 | best_period=1.86652 | Per=1.86657
# FLAG | star_idx=68 (12.482258,-73.047455) | angDist=15.4918 | best_period=3.22497 | Per=3.22495
# FLAG | star_idx=77 (12.588534,-73.102638) | angDist=15.4195 | best_period=5.89025 | Per=5.88951
# FLAG | star_idx=79 (12.620275,-73.120697) | angDist=13.8958 | best_period=5.95421 | Per=5.95478
# FLAG | star_idx=91 (12.826005,-72.723526) | angDist=16.0204 | best_period=6.38519 | Per=6.38458
# FLAG | star_idx=143 (13.350006,-73.11042) | angDist=14.8616 | best_period=3.9681 | Per=3.9681
# FLAG | star_idx=153 (13.42474,-72.908089) | angDist=13.2949 | best_period=8.32942 | Per=8.33199
# FLAG | star_idx=163 (13.495896,-72.262009) | angDist=18.8456 | best_period=3.16813 | Per=3.16815
# FLAG | star_idx=218 (14.087205,-72.84552) | angDist=17.1817 | best_period=4.85748 | Per=4.85752
# FLAG | star_idx=225 (14.127534,-72.231476) | angDist=13.5843 | best_period=1.66387 | Per=1.66391
# FLAG | star_idx=260 (14.733524,-71.976097) | angDist=18.3249 | best_period=7.05249 | Per=7.05486
# FLAG | star_idx=872 (12.422279,-73.237427) | angDist=7.92543 | best_period=15.6609 | Per=15.6574
# FLAG | star_idx=902 (14.603494,-72.42289) | angDist=15.2547 | best_period=1.52948 | Per=1.52954
# FLAG | star_idx=907 (13.615561,-72.266136) | angDist=16.9135 | best_period=4.24573 | Per=4.24593
# Done 1xJ/AcA/60/17/catalog classicalcepheids_smc: 17 flagged row(s) with angDist >= 2.0

# -------Processing 1xJ/AcA/61/103/ecl (eb_lmc)-----
# FLAG | star_idx=516 (75.956276,-68.934052) | angDist=13.1225 | best_period=25.8198 | Per=24.6098
# FLAG | star_idx=839 (88.22397,-69.48793) | angDist=14.9489 | best_period=9.1275 | Per=9.08157
# Done 1xJ/AcA/61/103/ecl eb_lmc: 2 flagged row(s) with angDist >= 2.0

# -------Processing 1xJ/AcA/63/323/ecl (eb_smc)-----
# FLAG | star_idx=181 (13.67278,-72.486137) | angDist=19.2702 | best_period=18.0434 | Per=18.2511
# FLAG | star_idx=193 (13.79491,-72.45266) | angDist=12.2267 | best_period=1.6707 | Per=1.6682
# Done 1xJ/AcA/63/323/ecl eb_smc: 2 flagged row(s) with angDist >= 2.0
# Total flagged rows: 29

## shannon note: also star 81, which is contaminated by a cepheid 26 arcsec away.

In [9]:
contaminants = [395, 444, 495, 790, 829, 839, 843, 1068, 44, 54, 65, 66, 68, 77, 79, 81, 91, 143, 153, 163, 218, 225, 260, 872, 902, 907, 516, 839, 181, 193]

# oct 2026

In [21]:
# -------Processing 1xJ/AcA/68/89/lmc_t2cep (typeIIcepheids_lmc)-----
# Done 1xJ/AcA/68/89/lmc_t2cep typeIIcepheids_lmc: 0 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/60/1/catalog (deltascut_lmc)-----
# Done 1xJ/AcA/60/1/catalog deltascut_lmc: 0 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/58/163/catalog (classicalcepheids_lmc)-----
# FLAG | star_idx=395 (72.485769,-69.755859) | angDist=15.3439 | best_period=27.1165 | Per=27.1006
# FLAG | star_idx=399 (72.593866,-69.332047) | angDist=13.8071 | best_period=2.23887 | Per=2.23906
# FLAG | star_idx=444 (74.058839,-67.413483) | angDist=8.88647 | best_period=3.60558 | Per=3.60559
# FLAG | star_idx=495 (75.158007,-67.764633) | angDist=18.506 | best_period=3.52438 | Per=3.52434
# FLAG | star_idx=829 (86.675929,-68.008675) | angDist=7.49653 | best_period=2.56664 | Per=2.56587
# Done 1xJ/AcA/58/163/catalog classicalcepheids_lmc: 5 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/59/1/catalog (RRlyrae_lmc)-----
# Done 1xJ/AcA/59/1/catalog RRlyrae_lmc: 0 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/59/239/catalog (lpv_lmc)-----
# FLAG | star_idx=504 (75.386393,-68.220688) | angDist=18.1629 | best_period=217.005 | Per=210.1
# FLAG | star_idx=1109 (82.58725,-69.457428) | angDist=12.8591 | best_period=104.299 | Per=108.38
# FLAG | star_idx=1124 (84.908046,-69.196167) | angDist=9.82583 | best_period=490.647 | Per=473.7
# Done 1xJ/AcA/59/239/catalog lpv_lmc: 3 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/61/217/lpv (lpv_smc)-----
# FLAG | star_idx=870 (12.100888,-73.225922) | angDist=8.91507 | best_period=30.0819 | Per=30.075
# Done 1xJ/AcA/61/217/lpv lpv_smc: 1 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/63/323/ecl (eb_smc)-----
# Done 1xJ/AcA/63/323/ecl eb_smc: 0 flagged row(s) with angDist >= 2.0
# -------Processing 1xJ/AcA/60/17/catalog (classicalcepheids_smc)-----
# FLAG | star_idx=902 (14.603494,-72.42289) | angDist=15.2547 | best_period=1.52942 | Per=1.52954
# FLAG | star_idx=218 (14.087205,-72.84552) | angDist=17.1817 | best_period=4.85732 | Per=4.85752
# FLAG | star_idx=173 (13.620808,-72.742271) | angDist=5.61273 | best_period=1.54471 | Per=1.5447
# FLAG | star_idx=907 (13.615561,-72.266136) | angDist=16.9135 | best_period=4.24604 | Per=4.24593
# FLAG | star_idx=65 (12.419711,-73.236916) | angDist=5.97339 | best_period=15.6585 | Per=15.6574
# Done 1xJ/AcA/60/17/catalog classicalcepheids_smc: 5 flagged row(s) with angDist >= 2.0
# Skipping 1xJ/AcA/60/165/ident (RRlyrae_smc): missing columns ['Per1']
# -------Processing 1xJ/AcA/61/103/ecl (eb_lmc)-----
# Done 1xJ/AcA/61/103/ecl eb_lmc: 0 flagged row(s) with angDist >= 2.0
# Total flagged rows: 14
# Wrote 14 flagged entries to ./contaminated_flagged_stars_oct26.csv
# WARNING: VerifyWarning: It is strongly recommended that column names contain only upper and lower-case ASCII letters, digits, or underscores for maximum compatibility with other software (got '<Imag>'). [astropy.io.fits.column]
# WARNING: VerifyWarning: It is strongly recommended that column names contain only upper and lower-case ASCII letters, digits, or underscores for maximum compatibility with other software (got '<Vmag>'). [astropy.io.fits.column]
# WARNING: VerifyWarning: It is strongly recommended that column names contain only upper and lower-case ASCII letters, digits, or underscores for maximum compatibility with other software (got '<Imag>1'). [astropy.io.fits.column]
# WARNING: VerifyWarning: It is strongly recommended that column names contain only upper and lower-case ASCII letters, digits, or underscores for maximum compatibility with other software (got '<Imag>2'). [astropy.io.fits.column]

In [24]:
contaminants_oct = [395, 399, 444, 495, 829, 504, 1109, 1124, 870, 902, 218, 173, 907, 65]

In [ ]:
# gaia table: J/A+A/606/A92